

---
## **How does the brain generate movement? Motor manifold discovery via dynamical systems**

---

Arthur Pellegrino, Isabel Cornacchia, Angus Chadwick | University of Edinburgh, School of Informatics | *Bio-inspired deep learning workshop*




## **Introduction**

Neuroscience studies how the brain processes information to produce behaviour, and studying how variables of the world are *represented* in neural activity is central to tackling this question. A topic of particular interest is the study of how the brain generates *movement*. This question is often explored by recording from monkeys performing movements and observing of their neural activity changes as a function of these movements.

**Task description.** In this project you will study neural recordings during a movement task. The subject -- a monkey -- is asked to move a cursor on the screen to reach a target by moving a joystick. While the monkey performed the task, neural activity was recorded from the primary motor cortex, an area of the brain involved in motor execution.

<p align="center">
  <img src="https://drive.google.com/uc?id=1Ouv2l8-8GcvsVBVttThsk_9OyQLFffjy" width="450">
</p>

**Figure 1.** **A.** A monkey was trained to move a joystick with their hand to reach a target on the screen.  **B.** In the meantime, populations of neurons were recorded in brain areas associated to movement.

**Pre-processing.** This task was repeated by the animal several hundreds of time. Each repetition is commonly called a "trial", each producing a neural data matrix of shape *time x neuron*, overall  making a 3D array (also called tensor) of shape *trial x time x neuron*. The targets on the screen were varied within a discrete set of 8 angles $\theta_{target}$, and we call such experimental variable of a task a "condition". The trials were averaged within each condition to produce a tensor of shape *condition x time x neuron*.

**Note.** This dataset is part of published work [1] but the data itself is not publicly available. We have shared part of it for the purpose of this project but please do not share.

## Loading the data

In [ ]:
import gdown
import numpy as np
import pickle
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt

# This will download the dataset stored in the Google drive
#!gdown 13uJppggakD-fe_doa1rQbVoWwvIn4s1a
#!pip install git+https://github.com/arthur-pe/mdds_workshop.git
#!pip install umap-learn
#!pip install tabulate
#!pip install optuna
#!pip install plotly
#!pip install torch


In [ ]:
def import_data(data_file, sigma=None):

    regions = ['M1']

    with open(data_file, 'rb') as f:
        data = pickle.load(f)

    neural_data = [data[r+'_spikes'] for r in regions]
    #print([[regions[i], neural_data[i].shape[-1]] for i, r in enumerate(regions)])
    neural_data = np.concatenate([data[r+'_spikes'] for r in regions], axis=-1)

    if sigma is not None:
        neural_data = gaussian_filter1d(neural_data.astype(float), axis=1, sigma=sigma)

    times = data['time']

    epochs = ['BL', 'AD', 'WO']

    trial_ids = []
    max_id = 0
    for e in epochs:
        trial_ids.append(np.array(data['trial_id'][data['epoch']==e])+max_id)
        max_id = trial_ids[-1][-1]

    trial_ids = np.concatenate(trial_ids)

    condition = data['condition']

    neural_data, condition = neural_data[data['epoch']=='BL'], condition[data['epoch']=='BL']

    condition = condition/(np.max(condition)+1)

    #neural_data = neural_data[:, (times<0.4) & (times>=-0.4)]


    neural_data = neural_data/neural_data.std()
    position_kinem=data['pos']
    #print(position_kinem.shape)
    #print("\n",np.sum(data['epoch']=='BL')," over ",(data['epoch']=='BL').shape);

    position_kinem=position_kinem[data['epoch']=='BL',:,:]
    #position_kinem=position_kinem[:, (times<0.4) & (times>=-0.4)]

    #times = times[(times<0.4) & (times>=-0.4)]

    neural_data = neural_data - neural_data[:, times<0.1].mean(axis=(0, 1))
    neural_data = neural_data / (neural_data.std(axis=(0, 1)) + 10 ** -6)
    return neural_data, condition, times,position_kinem

In [ ]:
data_file = 'reach.pkl'
data, condition, times,kin_pos = import_data(data_file, sigma=4)

Y = kin_pos

print("kin shape",kin_pos.shape)
print("neural data_shape ", data.shape)
print(data.shape, condition.shape, times.shape)

In [ ]:
print(kin_pos.shape)
classes=np.unique(condition)
for i in range(data.shape[0]):
    class_color=np.where(classes==condition[i])[0][0]
    clr=plt.cm.tab10(class_color)
    plt.plot(Y[i,:,0],Y[i,:,1], label="kin pos "+str(i),color=clr,linewidth=0.5)


## Data information

The neural data consists of a tensor of shape `condition x time x neuron = 218 x 80 x 95`. The condition is an array containing a label for the angle of the target that the animal was instructed to reach (8 unique values $\in[0,1]$).

In [ ]:
angle_order = np.unique(condition)
angle_order


In [ ]:
color = np.outer(condition, np.ones((data.shape[1]))).flatten()

# Position Trajectories

In [ ]:
Y_stacked = np.vstack(Y)
Y_stacked.shape, color.shape

In [ ]:
# prompt: do a 3d scatter plot of X_pca with matplotlib

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

plt.scatter(Y_stacked[:, 0], Y_stacked[:, 1], c=color, alpha=0.5, s=2,cmap='hsv')

## PCA

First, to get a sense of the structure of the data, apply PCA to them.

* To the last time point i.e. a slice of your tensor of shape *condition x neuron*. Make a scatter plot in a 2D PCA space.
* To the full data by unfolding the tensor into a *(condition x time) x neuron* matrix. Make a plot of the trajectories in a 3D PCA space.

If you color the trajectories according to the condition, what do you observe?

In [ ]:
X_stacked = np.vstack(data)
U, S, Vt = np.linalg.svd(X_stacked, full_matrices=False)
X_pca = U[:, :3] @ np.diag(S[:3])
X_pca.shape, condition.shape

In [ ]:
X_pca_FULL = U[:, :] @ np.diag(S[:])


In [ ]:
X_stacked = np.vstack(data)
U, S, Vt = np.linalg.svd(X_stacked, full_matrices=False)
X_pca30 = U[:, :30] @ np.diag(S[:30])
X_pca30.shape, condition.shape

In [ ]:
# prompt: do a 3d scatter plot of X_pca with matplotlib

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2],s = 0.15, c = color)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
plt.show()


In [ ]:
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2],s = 0.15, c = color)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.view_init(elev=0, azim=0)  # elevation and azimuth angles

plt.show()



In [ ]:
X_last_time = X_pca.reshape(data.shape[0], data.shape[1], -1).mean(axis=1)

X_last_time.shape

In [ ]:
X_pca.shape
X_pca_reshape=X_pca.reshape((data.shape[0],data.shape[1],-1));
X_pca_reshape.shape
for i in range(data.shape[0]):
  colo="C"+str(int(condition[i]*8))
  plt.plot(X_pca_reshape[i,-1,1],X_pca_reshape[i,-1,2],c=colo,marker="o")
plt.show()


In [ ]:
import umap.umap_ as umap
um = umap.UMAP(
    n_components=2,
    n_neighbors=30,
    min_dist=0.0,#0.01,
    metric='euclidean',
    verbose=True
)
X_umap = um.fit_transform(X_stacked)

In [ ]:
X_umap_0 = um.transform(data[0,:,:])
X_umap_1 = um.transform(data[1,:,:])

In [ ]:
# prompt: Plot X_umap in a 2 d plot

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(X_umap[:, 0], X_umap[:, 1], c=color, s=10)
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.title("UMAP Projection of Neural Data")
plt.colorbar(label="Condition")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(X_umap_1[:, 0], X_umap_1[:, 1], s=10)
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.title("UMAP Projection of Neural Data")
plt.colorbar(label="Condition")
plt.show()

## MDDS application

The results of PCA are limited in two key ways:
1. They assume that the neural representation of time lives in a flat subspace, as opposed to a curved manifold.
2. They don't provide hints as to the *computations* that are performing by the brain on this representation. In particular, we still don't know the dynamical systems underlying the data.

To address this you'll fit MDDS to the data. You can refer to our [preprint](https://drive.google.com/file/d/1rTJfntd2tC6JkylfUdq6UeehjXiFNAnS/view?usp=drive_link) for the details of the method. The key insight of the method is that near a point on the manifold, the *dynamics*, that is the time derivative of the neural trajectories $\frac{d\mathbf{x}(t)}{dt}:=\mathbf{\dot x}(t)$, are constrained to the vector space that is tangent to the manifold. Conversely, the manifold is defined by the tangent space along with a distinguished point. Therefore, parameterizing dynamics on a manifold is equivalent to dynamically choosing a vector of the tangent space to move along:

$$
  \mathbf{\dot x} = c_1(t)\mathbf{f}_1(\mathbf{x}) + ... + c_k(t)\mathbf{f}_k(\mathbf{x})  
$$
where $\mathbf{x}(t)\in \mathbb{R}^n$ is the state of the system, $\mathbf{f}_i:\mathbb{R}^n \rightarrow \mathbb{R}^n$ a vector field for $n$ neurons$^{*}$, and $k$ is the dimension of the tangent space (and therefore that of the manifold).

We can then fit the parameters of the model (find the optimal $c$'s and $\mathbf{f}'s$) to minimize the distance between the datapoints and points along the trajectories of the model.

<p align="center">
  <img src="https://drive.google.com/uc?id=1V9wL8vDn2lNqQYeu-0GjiNDuaJnDcNMn" width="400">
</p>

**Figure 2: Manifold discovery via dynamical systems.** (Left) trajectories on a manifold have derivatives constrained to the space that is tangent to the manifold. (Right) the manifold is inferred by fitting the model parameters to minimize the distance between the trajectories and data points.

For the sake of having a more interpretable model, you can use *linear vector fields*. That is:
$$
  \mathbf{f}_i(\mathbf{x}) = W_i\mathbf{x}(t)
$$
where $\mathbf{W}_i\in\mathbb{R}^{n \times n}$ is a matrix describing the interactions between the variables of the system. There are other parametrisations available if you want to relax this assumption.

**Optional.** In fact, if the manifold is embedded within a lower-dimensional subspace of the space of the recorded neurons, we may need fewer than $n$ variables to describe the activity of the $n$ recorded neurons. For example, we could choose to have an $m$-dimensional latent space and decode from it as $\mathbf{y}(t)=D\mathbf{x}(t)$ where $D\in \mathbb{R}^{n \times m}$ and $\mathbf{y}(t)$ is the data.

* Apply MDDS (already done for you below)
* Compute how much variance MDDS explains as a function of the manifold dimensionality (you can access the data estimate in `data_estimate.npy`)
* Compare that to how much variance a subspace manifold explains as a function of its dimensionality
* Try to explain why $2$ might be a theoretically sound dimensionality for the manifold

In [ ]:

from IPython.display import clear_output

clear_output()

import mdds
from mdds import utils

import matplotlib

As in most ML methods, there are a lot of hyperparameters. We've set them to reasonable values for you.

In [ ]:
p = {

'seed' : 33,

'save_dir': utils.make_directory(), # Where the model etc... is saved

'manifold_dim': 2, # Intrinsic dim (to be renamed)
'embedding_dim' : 12, # Embedding dim (MDDS's dynamics can be of a different dimension than the number of data neurons).

'frobenius_penalty': 0.03, # Loss = ||data - mdds|| + frobenius_penalty*sum(LieBrackets)

'batch_prop': 0.6, # What fraction of the data to use in each iteration (i.e. minibatch)
'max_iterations': 1800,

'learning_rate': 0.03, # 0.05 - 0.0001
'weight_decay': 0.001, # Regularization on the model's parameters (0, 0.1 or 1.0)
'min_std_test_loss': 0.001,
'min_test_iterations': 100, # If std of test_loss over the past min_test_iterations is less than min_std_test_loss then exit.

'test_frequency': 100, # Number of iterations in-between each plots update
'mesh_size': 61, # Plotted manifold mesh size

# ODE solver hyperparameters (see Driffrax)
'rtol': 0.01,
'atol': 0.001,

# If data has T time bins then the controls (i.e. c's) will be a linear interpolation of controls_interpolation_freq * T points
'controls_interpolation_freq': 1.0,

# The mode parameterizing the vector fields
'vector_fields_class': 'LinearMDDS_', # or DNNMDDS for a nonlinear vector field, in which case:
#'vector_fields_hyperparameters':{
#  'mlp_width': 10,
#  'mlp_depth': 2,
#  'activation': jnp.tanh
#},

'optimized': ['controls', 'vector_fields', 'decoder'], # Which of ['controls', 'vector_fields', 'decoder'] to optimize

}

In [ ]:
data.shape

In [ ]:
#mdds.fit(data, condition, cmap_condition=matplotlib.colormaps['hsv'], **p)

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

## Analysis of the MDDS solution
A common question in neuroscience is: how much information about a variable (e.g., a stimulus or behavioral parameter) is encoded in neural activity?

One way to approach this is through decoding — trying to predict the variable of interest from the neural responses.

* Linearly decode the target angle from the neural activity (e.g. at the final time point)

This will tell us how well the neural population represents the angle, using only a linear readout.

* Repeat this with the activity from MDDS (`latent_factors.npy` or `data_estimate.npy`). Is there one representation that allows better (linear) decoder? What would this imply for the system?

## Classification of Target ANGLE

In [ ]:
data.shape
raw_data_last_time=data[:,-1,:].squeeze()
raw_data_last_time.shape
print(np.unique(condition))

X_pca_last_time_selected=X_pca_reshape[:,-1,1:3]
X_pca_last_time_selected.shape

X_umap_last_time=X_umap.reshape((data.shape[0],data.shape[1],-1));
X_umap_last_time=X_umap_last_time[:,-1,:].squeeze()
X_umap_last_time.shape

X_latent_MDDS=np.load("runs/all/24-04-2025_15_47_21/latent_factors.npy")
X_latent_MDDS_stacked = np.vstack(X_latent_MDDS)
X_latent_MDDS_last_time=X_latent_MDDS[:,-1,:].squeeze()
print(X_latent_MDDS_last_time.shape)#

X_data_MMDS=np.load("runs/all/24-04-2025_15_47_21/data_estimate.npy")
X_data_MDDS_stacked = np.vstack(X_data_MMDS)
X_data_MMDS_last_time=X_data_MMDS[:,-1,:].squeeze()
print(X_data_MMDS_last_time.shape)

In [ ]:
data.shape

In [ ]:
#!pip install tabulate
from tabulate import tabulate
from sklearn.model_selection import StratifiedKFold
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import matthews_corrcoef
import numpy as np

# Define stratified K-fold


kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=10)
classes=np.array([np.argwhere(np.unique(condition) == e).flatten()[0] for e in condition])
#print(classes)
# Store accuracy per fold
matt_coeffs =[]
matt_coeffs_items=[]
for X,name in [(raw_data_last_time,"RAW 95-D"),
               (X_pca_last_time_selected,"PCA 2-D"),
               (X_umap_last_time,"UMAP 2-D"),
               (X_latent_MDDS_last_time,"MDDS LATENT c(t) 2-D"),
               (X_data_MMDS_last_time,"MDDS GENERATED_DATA y(t) 95-D")
               ]:
  l=[];
  for train_idx, val_idx in kfold.split(X, classes):
      lda = LDA(n_components=None)
      #print(X[train_idx])
      lda.fit(X[train_idx,:], classes[train_idx])
      predictions = lda.predict(X[val_idx,:])

      y_val=classes[val_idx]

      c_m = matthews_corrcoef(y_val, predictions)
      l.append(c_m)
  matt_coeffs.append((name,np.mean(l),np.std(l)))
  matt_coeffs_items.append((name,l.copy()))
print("CLASSIFICATION RESULTS")
print(tabulate(matt_coeffs, ["Model name", "Matthews Coeffs[-1,1]","std Matthews coeffs"], tablefmt="github"))



In [ ]:
import matplotlib.pyplot as plt

# Sort matt_coeffs_items by the mean value of the coefficients
sorted_items = sorted(matt_coeffs_items, key=lambda x: np.mean(x[1]), reverse=True)

# Extract names and values
names = [item[0] for item in sorted_items]
values = [item[1] for item in sorted_items]

# Create the box plot
plt.figure(figsize=(10, 6))
plt.boxplot(values, vert=False, patch_artist=True, labels=names)
plt.xlabel("Matthews Coefficient")
plt.title("Box Plot of Matthews Coefficients (Sorted by Mean)")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.show()

## Regression Model

Regression from position to dim reduction latent variables

In [ ]:
#X_umap_averaged = np.vstack([X_umap[color==i].reshape(-1,data.shape[1],X_umap.shape[-1]).mean(axis=0) for i in np.unique(color)])
#X_pca_averaged = np.vstack([X_pca[color==i].reshape(-1,data.shape[1],X_pca.shape[-1]).mean(axis=0) for i in np.unique(color)])
#X_averaged = np.vstack([X_stacked[color==i].reshape(-1,data.shape[1],X_stacked.shape[-1]).mean(axis=0) for i in np.unique(color)])
#X_latent_MDDS_averaged = np.vstack([X_latent_MDDS_stacked[color==i].reshape(-1,data.shape[1],X_latent_MDDS_stacked.shape[-1]).mean(axis=0) for i in np.unique(color)])
#X_data_MDDS_averaged = np.vstack([X_data_MDDS_stacked[color==i].reshape(-1,data.shape[1],X_data_MDDS_stacked.shape[-1]).mean(axis=0) for i in np.unique(color)])
#Y_averaged = np.vstack([Y_stacked[color==i].reshape(-1,data.shape[1],Y_stacked.shape[-1]).mean(axis=0) for i in np.unique(color)])

### Averaged Regression

In [ ]:
"""
#!pip install tabulate
from tabulate import tabulate
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import numpy as np
import pandas as pd

# Define stratified K-fold

R2_scores = []
kfold = KFold(n_splits=5, shuffle=True, random_state=10)
for name, X__ in [
    ("umap 2",X_umap_averaged),
    ("pca 3",X_pca_averaged), 
    ("neural 95",X_averaged),
    ("mdds 95", X_data_MDDS_averaged),
    ("mdds 2", X_latent_MDDS_averaged) 
    ]:
    X_ = np.vstack(X__)
    for train_idx, val_idx in kfold.split(X_):
        X_train = np.hstack([X_[train_idx, :], np.ones((X_[train_idx, :].shape[0], 1))])
        X_val = np.hstack([X_[val_idx, :], np.ones((X_[val_idx, :].shape[0], 1))])
        y_train = Y_averaged[train_idx]
        y_val = Y_averaged[val_idx]

        A = np.linalg.solve(X_train.T @ X_train, X_train.T @ y_train)
        y_pred = X_val @ A

        r2 = r2_score(y_val, y_pred)
        R2_scores.append({"name": name, "R2": r2})

R2_scores = pd.DataFrame(R2_scores)
R2_scores = R2_scores.groupby(["name"]).agg(["mean", "std"]).reset_index()
R2_scores.sort_values(by=("R2", "mean"), ascending=False, inplace=True)
R2_scores"""

### Not Averaged Regression

In [ ]:
#!pip install tabulate
from tabulate import tabulate
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
# Define stratified K-fold

R2_scores = []
kfold = KFold(n_splits=5, shuffle=True, random_state=10)
for name, X_ in [
    ("umap 2",X_umap),
    ("pca 3",X_pca), 
    ("neural 95",X_stacked),
    ("mdds 95", X_data_MDDS_stacked),
    ("mdds 2", X_latent_MDDS_stacked) 
    ]:
    for train_idx, val_idx in kfold.split(X_):
        X_train =X_[train_idx,:];# np.hstack([X_[train_idx, :], np.ones((X_[train_idx, :].shape[0], 1))])
        X_val =X_[val_idx,:]# np.hstack([X_[val_idx, :], np.ones((X_[val_idx, :].shape[0], 1))])
        y_train = Y_stacked[train_idx]
        y_val = Y_stacked[val_idx]
        
        model=LinearRegression()
        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)

        #A = np.linalg.solve(X_train.T @ X_train, X_train.T @ y_train)
        #y_pred = X_val @ A

        r2 = r2_score(y_val, y_pred)
        R2_scores.append({"name": name, "R2": r2})

R2_scores = pd.DataFrame(R2_scores)
R2_scores = R2_scores.groupby(["name"]).agg(["mean", "std"]).reset_index()
R2_scores.sort_values(by=("R2", "mean"), ascending=False, inplace=True)
print("REGRESSION RESULTS")
R2_scores

## R^2 varying # retained PCs

In [ ]:

R_means=    []
dimension_pcs=    []
R_stds=    []
for i_pcs in range(2,X_pca_FULL.shape[1]):
    X_=X_pca_FULL[:,:i_pcs]
    R2_scores = []
    kfold = KFold(n_splits=5, shuffle=True, random_state=10)
    for train_idx, val_idx in kfold.split(X_):
        X_train = X_[train_idx,:]#np.hstack([X_[train_idx, :], np.ones((X_[train_idx, :].shape[0], 1))])
        X_val = X_[val_idx,:]#np.hstack([X_[val_idx, :], np.ones((X_[val_idx, :].shape[0], 1))])
        y_train = Y_stacked[train_idx]
        y_val = Y_stacked[val_idx]

        model=LinearRegression()
        model.fit(X_train, y_train)
        y_pred=model.predict(X_val)
        #print(y_pred.shape,y_val.shape,X_val.shape,X_train.shape)
        #A = np.linalg.solve(X_train.T @ X_train, X_train.T @ y_train)
        #y_pred = X_val @ A
        R2_scores.append(r2_score(y_val, y_pred))
    R_means.append(np.mean(R2_scores))
    R_stds.append(np.std(R2_scores))
    dimension_pcs.append(i_pcs)

# Adding jitter to the points
jitter = 0.1
jittered_dimensions = np.array(dimension_pcs) #+ np.random.uniform(-jitter, jitter, len(dimension_pcs))

# Plotting the mean R^2 with shading for ±2 std
plt.figure(figsize=(10, 6))
plt.plot(dimension_pcs, R_means, label="Mean R^2", color="blue", linewidth=2)
plt.fill_between(dimension_pcs, 
                 np.array(R_means) - 2 * np.array(R_stds), 
                 np.array(R_means) + 2 * np.array(R_stds), 
                 color="blue", alpha=0.2, label="±2 Std Dev")

# Adding jittered points
#plt.scatter(dimension_pcs, R_means, color="red", alpha=0.6, label="Jittered Points")

# Adding labels and legend
plt.xlabel("Number of Principal Components", fontsize=12)
plt.ylabel("R^2 Score", fontsize=12)
plt.title("R^2 Score vs Number of Principal Components", fontsize=14)
plt.legend()
plt.grid(True)
plt.show()


# UMAP hyperparameter optimization

In [ ]:
import optuna
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
import numpy as np
from sklearn.linear_model import LinearRegression
import umap.umap_ as umap

A_best = None
R2_best = -np.inf
model_best=None
# Define the objective function for Optuna
def objective(trial):
    global A_best, R2_best,model_best
    # Suggest hyperparameters
    n_neighbors = trial.suggest_int("n_neighbors", 5, 50)
    min_dist = trial.suggest_float("min_dist", 0.0, 0.5)
    n_components = trial.suggest_int("n_components", 2, 10)

    # Initialize UMAP with suggested hyperparameters
    um = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        n_components=n_components,
        metric="euclidean"
    )

    # Perform K-Fold cross-validation
    kfold = KFold(n_splits=5, shuffle=True)
    r2_scores = []
    model_s = []

    for train_idx, val_idx in kfold.split(X_stacked):
        X_train, X_val = X_stacked[train_idx], X_stacked[val_idx]
        Y_train, Y_val = Y_stacked[train_idx], Y_stacked[val_idx]

        # Fit UMAP on training data
        X_train_umap = um.fit_transform(X_train)
        X_val_umap = um.transform(X_val)

        # Linear regression to predict Y from UMAP-transformed X
        #X_train_umap = np.hstack([X_train_umap, np.ones((X_train_umap.shape[0], 1))])
        #X_val_umap = np.hstack([X_val_umap, np.ones((X_val_umap.shape[0], 1))])

        model= LinearRegression()
        model.fit(X_train_umap, Y_train)
        Y_pred = model.predict(X_val_umap)
#        A = np.linalg.solve(X_train_umap.T @ X_train_umap, X_train_umap.T @ Y_train)
#        Y_pred = X_val_umap @ A

        # Compute R2 score
        r2 = r2_score(Y_val, Y_pred)
        r2_scores.append(r2)
        model_s.append(model)


    mean_r2 = np.mean(r2_scores)
    #Saving best A and R2
    if mean_r2 > R2_best:
        R2_best = mean_r2
        model_best =model_s[np.argmax(r2_scores)]

    # Return the mean R2 score across folds
    return np.mean(r2_scores)

In [ ]:
#study = optuna.create_study(direction="maximize", study_name="UMAP Hyperparameter Optimization", storage="sqlite:///umap_search_withSCIKIT.db", load_if_exists=True)

In [ ]:
#study.optimize(objective, n_trials=20, n_jobs=10)

In [ ]:
#best_param= {"n_neighbors":29,"min_dist":0.14715882726479382,"n_components":10};
#best_param={"n_neighbors":study.best_params[""],"min_dist":0.14715882726479382,"n_components":10};
import pickle
#with open("best_param_vSCIKIT.pkl", "wb") as f:
#    pickle.dump(study.best_params, f)
#with open("best_Lin_Regr_model_vSCIKIT.pkl", "wb") as f:
#    pickle.dump(model_best, f)
#with open("best_R2_vSCIKIT.pkl", "wb") as f:
#    pickle.dump(R2_best, f)
with open("best_param_vSCIKIT.pkl", "rb") as f:
    best_params = pickle.load(f)
with open("best_Lin_Regr_model_vSCIKIT.pkl", "rb") as f:
    model_best = pickle.load(f)
with open("best_R2_vSCIKIT.pkl", "rb") as f:
    R2_best = pickle.load(f)

In [ ]:
print("Best hyperparameters:", best_params)
model_best.coef_.shape

In [ ]:
# Apply UMAP with the best hyperparameters
best_umap = umap.UMAP(
    n_neighbors=best_params["n_neighbors"],
    min_dist=best_params["min_dist"],
    n_components=best_params["n_components"],
    metric="euclidean",
    random_state=11
)
X_umap_best = best_umap.fit_transform(X_stacked)

In [ ]:
# Plot the 2D scatter plot with tab10 discrete color scheme
plt.figure(figsize=(8, 6))
plt.scatter(X_umap_best[:, 1], X_umap_best[:, 2], c=color, s=1, cmap='tab10')
plt.xlabel("UMAP Dimension 2")
plt.ylabel("UMAP Dimension 3")
plt.title("2D UMAP Projection with Best Hyperparameters")
plt.colorbar(label="Class")
plt.show()



In [ ]:
print(X_umap_best.shape)


In [ ]:
import pandas as pd

kfold = KFold(n_splits=5, shuffle=True)
r2_scores = []

X_umap_with_biases = X_umap_best#np.hstack([X_umap_best, np.ones((X_umap_best.shape[0], 1))])

Y_pred =model_best.predict( X_umap_with_biases)

# Plot the 2D scatter plot with tab10 discrete color scheme
fig, axs = plt.subplots(1, 2, figsize=(12, 6),sharex=True, sharey=True)
axs[0].scatter(Y_pred[:, 0], Y_pred[:, 1], c=color, s=3, cmap='tab10', label="Prediction", marker='o')
axs[0].set_title("Predicted Trajectories")
axs[0].set_xlabel("X")
axs[0].set_ylabel("Y")
axs[0].legend()

axs[1].scatter(Y_stacked[:, 0], Y_stacked[:, 1], c=color, s=4, cmap='tab10', label="Original", marker='s')
axs[1].set_title("Original Trajectories")
axs[1].set_xlabel("X")
axs[1].set_ylabel("Y")
axs[1].legend()

plt.suptitle("2D Kinematic and prediction from Linear Regressions using UMAP dimensionality reduction")
plt.show()
#plt.scatter(Y_stacked[:, 0], Y_stacked[:, 1], c=color, s=4, cmap='tab10', label="Original ", marker='s')
#
##markerfacecolor='none', markeredgecolor='blue', for open cirles
#plt.xlabel("X")
#plt.ylabel("Y")
#plt.title("2D Kinematic and prediction from Linear Regressions using UMAP dimensionality reduction")
#plt.colorbar(label="Class")
#plt.legend()
#plt.show()



## Nonlinear Neural Regression

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Model(torch.nn.Module):
    def __init__(self, input_dim, output_dim):
        super(Model, self).__init__()
        self.model = torch.nn.Sequential(
            torch.nn.Linear(input_dim, input_dim // 2),  # First layer
            torch.nn.Sigmoid(),
            torch.nn.Linear(input_dim // 2, input_dim // 4),  # Second layer
            torch.nn.Sigmoid(),
            torch.nn.Linear(input_dim // 4, output_dim),  # Third layer
            torch.nn.Sigmoid()
        )

    def forward(self, x):
        return 2 * self.model(x) - 1

scaler = MinMaxScaler()
Y_normalized = scaler.fit_transform(Y_stacked)

In [ ]:
Y_train, Y_test, X_train, X_test, _, c_test = train_test_split(Y_normalized, X_stacked, color, test_size=0.2, random_state=42)

with device:
    loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32).to(device), torch.tensor(Y_train, dtype=torch.float32).to(device)), batch_size=512, shuffle=True)
"""
    model_mlp = Model(X_train.shape[1], Y_train.shape[1]).to(device)
    optimizer = Adam(model_mlp.parameters(), lr=0.01)
    criterion = torch.nn.MSELoss()

    for epoch in range(150):
        model_mlp.train()
        for batch_X, batch_Y in loader:
            optimizer.zero_grad()
            predictions = model_mlp(batch_X)
            loss = criterion(predictions, batch_Y)
            loss.backward()
            optimizer.step()

        if epoch % 10 == 0:
            model_mlp.eval()
            with torch.no_grad():
                test_predictions = model_mlp(torch.tensor(X_test, dtype=torch.float32))
                test_loss = criterion(test_predictions, torch.tensor(Y_test, dtype=torch.float32))
                print(f"Epoch {epoch}, Test Loss: {test_loss.item()}")
"""

In [ ]:
model_mlp=Model(X_train.shape[1], Y_train.shape[1]).to(device)  # Initialize the model
model_mlp.load_state_dict(torch.load('model_mlp.pth'))  # load saved weights
Y_pred = model_mlp(torch.tensor(X_test, dtype=torch.float32)).detach().cpu().numpy()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 6),sharex=True, sharey=True)
axs[0].scatter(Y_pred[:, 0], Y_pred[:, 1], c=c_test, s=3, cmap='tab10', label="Prediction", marker='o')
axs[0].set_title("Predicted Datapoints") 
axs[0].set_xlabel("X")
axs[0].set_ylabel("Y")
axs[0].legend()
axs[1].scatter(Y_test[:, 0], Y_test[:, 1], c=c_test, s=4, cmap='tab10', label="Original", marker='s')
axs[1].set_title("Original Datapoints")
axs[1].set_xlabel("X")
axs[1].set_ylabel("Y")
axs[1].legend()
plt.suptitle("")
plt.show()

In [ ]:
torch.save(model_mlp.state_dict(), "model_mlp.pth")

## Direct Linear Regression

In [ ]:
font_settings = {'fontsize': 14, 'fontname': 'Arial'}
plt.rcParams.update({
    'font.size': font_settings['fontsize'],
    'font.family': font_settings['fontname']
})
lr_direct = LinearRegression()
lr_direct.fit(X_stacked, Y_stacked)
Y_pred_direct = lr_direct.predict(X_stacked)

# Plot the 2D scatter plot with tab10 discrete color scheme
fig, axs = plt.subplots(1, 2, figsize=(12, 6),sharex=True, sharey=True)
axs[0].scatter(Y_pred_direct[:, 0], Y_pred_direct[:, 1], c=color, s=3, cmap='tab10', label="Prediction", marker='o')
axs[0].set_title("Predicted Trajectories") 
axs[0].set_xlabel("X")
axs[0].set_ylabel("Y")
axs[0].legend()
axs[1].scatter(Y_stacked[:, 0], Y_stacked[:, 1], c=color, s=4, cmap='tab10', label="Original", marker='s')
axs[1].set_title("Original Trajectories")
axs[1].set_xlabel("X")
axs[1].set_ylabel("Y")
axs[1].legend()
plt.suptitle("Linear Regressions using Neural-95D")
plt.show()

## Linear Dynamical System

In [ ]:
def reshape_tensors(tensor):
    """
    Reshapes the input tensor into two tensors: tensor_minus and tensor_plus.

    Parameters:
        tensor (numpy.ndarray): Input tensor of shape (n_trials, TIME, n_features).

    Returns:
        tuple: A tuple containing tensor_minus and tensor_plus.
    """
    tensor_minus = np.zeros((tensor.shape[0] * (tensor.shape[1] - 1), tensor.shape[2]))
    tensor_plus = np.zeros((tensor.shape[0] * (tensor.shape[1] - 1), tensor.shape[2]))
    offset = 0

    for i_trial in range(tensor.shape[0]):
        this_trial = tensor[i_trial, :]
        tensor_minus[offset:offset + this_trial.shape[0] - 1, :] = this_trial[:-1, :].copy()
        tensor_plus[offset:offset + this_trial.shape[0] - 1, :] = this_trial[1:, :].copy()
        offset += this_trial.shape[0] - 1

    return tensor_minus, tensor_plus

In [ ]:
X_by_classes = [data[classes == i, :, :] for i in range(classes.max() + 1)]

In [ ]:
np.unique(condition)

In [ ]:
np.unique(classes)

In [ ]:
Ws = []
Ys = []
Xs = []
for i in range(classes.max() + 1):
    X_minus, X_plus = reshape_tensors(X_by_classes[i])
    W = np.linalg.solve(X_minus.T @ X_minus, X_minus.T @ X_plus)
    X_minus = X_minus.reshape(-1, data.shape[1] - 1, data.shape[2])
    X_plus = X_plus.reshape(-1, data.shape[1] - 1, data.shape[2])
    Ws.append(W)
    X0 = X_by_classes[i][:, 0, :].reshape(-1, data.shape[2])
    X_evol_list = []
    Y_evol_list = []
    for j in range(X_minus.shape[1]):
        X_evol = X_minus[:, j, :] @ W
        X_evol_list.append(X_evol)
        Y_evol = model_mlp(torch.from_numpy(X_evol).float()).detach().cpu().numpy()
        Y_evol_list.append(Y_evol)
    Ys.append(np.hstack(Y_evol_list)) # vstack
    Xs.append(np.hstack(X_evol_list)) # vstack

In [ ]:
Y_normalized.reshape(data.shape[0],-1,2).shape, np.vstack(Ys).reshape(-1, data.shape[1] - 1, 2).shape

In [ ]:
print("ok")

In [ ]:
ax, fig = plt.subplots(1, 2, figsize=(12, 6),sharex=True, sharey=True)
a = np.unique(condition)
for i, Y_result in enumerate(Ys):
    print(Y_result.shape)
    Y_result = Y_result.reshape(-1, data.shape[1] - 1, 2)
    for j in range(Y_result.shape[0]):
        strimg="C"+str(i)
        fig[1].scatter(Y_result[j, :, 0], Y_result[j, :, 1], c=strimg, linewidth=0.5,s=0.5)
        fig[1].set_title("Predicted Kinematic Trajectories")

Y_normalized_reshaped = Y_normalized.reshape(data.shape[0],-1,2)
c = np.unique(classes)
for i in range(Y_normalized_reshaped.shape[0]):
    strimg="C"+str(classes[i])
    fig[0].scatter(Y_normalized_reshaped[i, :, 0], Y_normalized_reshaped[i, :, 1], c=strimg, label="Original", linewidth=0.5,s=0.5)
    fig[0].set_title("Original Kinematic Trajectories")

plt.show()

In [ ]:
ax, fig = plt.subplots(1, 2, figsize=(12, 6),sharex=True, sharey=True)
a = np.unique(condition)
for i, Y_result in enumerate(Ys):
    print(Y_result.shape)
    Y_result = Y_result.reshape(-1, data.shape[1] - 1, 2)
    for j in range(Y_result.shape[0]):
        strimg="C"+str(i)
        fig[1].plot(Y_result[j, :, 0], Y_result[j, :, 1], c=strimg, linewidth=0.5)
        fig[1].set_title("Predicted Kinematic Trajectories")

Y_normalized_reshaped = Y_normalized.reshape(data.shape[0],-1,2)
c = np.unique(classes)
for i in range(Y_normalized_reshaped.shape[0]):
    strimg="C"+str(classes[i])
    fig[0].plot(Y_normalized_reshaped[i, :, 0], Y_normalized_reshaped[i, :, 1], c=strimg, label="Original", linewidth=0.5)
    fig[0].set_title("Original Kinematic Trajectories")

plt.show()

In [ ]:
ax, fig = plt.subplots(1, 2, figsize=(12, 6),sharex=True, sharey=True)
a = np.unique(condition)
for i, Y_result in enumerate(Ys):
    Y_result = Y_result.reshape(-1, data.shape[1] - 1, 2)
    for j in range(Y_result.shape[0]):
        strimg="C"+str(i)
        fig[1].plot(Y_result[j, :, 0], Y_result[j, :, 1], c=strimg, linewidth=0.5)
        fig[1].set_title("Predicted Trajectories")

Y_normalized_reshaped = Y_normalized.reshape(data.shape[0],-1,2)
c = np.unique(classes)
for i in range(Y_normalized_reshaped.shape[0]):
    strimg="C"+str(classes[i])
    fig[0].plot(Y_normalized_reshaped[i, :, 0], Y_normalized_reshaped[i, :, 1], c=strimg, label="Original", linewidth=0.5)
    fig[0].set_title("Original Trajectories")

plt.show()

In [ ]:
X_minus,X_plus=
W=np.linalg.solve(X_minus.T @ X_minus, X_minus.T @ X_plus)
X0 = data[:,0,:]
T_GEN = data.shape[1]

X_result= np.zeros((X0.shape[0],T_GEN, X0.shape[1]))
X_result[:,0,:] = X0.copy()
for i_T in range(1,T_GEN):
    X_result[:,i_T,:] = X_result[:,i_T-1,:] @ W.T    
plt.figure(figsize=(8, 6))

for i in range(X_result.shape[0]):  
    colore_trial="C"+str(classes[i])
    plt.scatter(X_result[i,:, 0], X_result[i,:, 1], c=colore_trial, s=1)
    plt.xlabel("X_neural Dimension 1")
    plt.ylabel("X_neural Dimension 2")
    plt.title("X_neural Projection of Neural Data Without external control")


In [ ]:
Y_result = model_mlp(torch.from_numpy(np.vstack(X_result)).float()).detach().cpu().numpy()

In [ ]:
plt.scatter(Y_result[:, 0], Y_result[:, 1], c=color, s=1)

## Big question: how is the brain solving this task?
The ultimate goal is to form a hypothesis for how the brain solves this task.

What is a (minimal) computational model that could explain the data?

There are many ways to address this. For example, you could start from the data and the properties extracted with the exploratory analysis (e.g. PCA). Or, you could consider the data and insights extracted via MDDS to inform your model. Alternatively, a common approach in neuroscience is to train a recurrent neural network (RNN) - which is a dynamical system - to solve the same task and compare that solution with your data and gain some insights into how the task *could* be solved.

How much can any of these approaches really inform us about the brain? How much does the available data constrains our models? For example, what can we *not* say about this system given this data?

**Other potential questions**

- For BCI application, consider a fixed *decoder*. How would it change the solution?

## References

[1] Perich, Matthew G., Juan A. Gallego, and Lee E. Miller. "A neural population mechanism for rapid learning." Neuron 100.4 (2018): 964-976.